# Fine-grained SGG Evaluation Analysis

读取 `eval_results.pytorch` 和 `result_dict.pytorch`，做谓词级召回、图像级失败案例、score 分布、relationness 影响等细粒度分析。

使用方式：先在下面配置 `EVAL_RESULTS_PATH` 和 `RESULT_DICT_PATH`。如果留空，notebook 会在仓库下自动搜索。

In [ ]:
import os
import sys
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'maskrcnn_benchmark').exists():
    REPO_ROOT = Path('/Users/shangfei/Developer/SDSGG')
sys.path.insert(0, str(REPO_ROOT))

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 180)
plt.rcParams['figure.figsize'] = (12, 4)

# 手动填路径；留空则自动搜索最新文件。
EVAL_RESULTS_PATH = ''
RESULT_DICT_PATH = ''


In [ ]:
VG_REL_CLASSES = [
    '__background__', 'above', 'across', 'against', 'along', 'and', 'at', 'attached to', 'behind',
    'belonging to', 'between', 'carrying', 'covered in', 'covering', 'eating', 'flying in', 'for',
    'from', 'growing on', 'hanging from', 'has', 'holding', 'in', 'in front of', 'laying on',
    'looking at', 'lying on', 'made of', 'mounted on', 'near', 'of', 'on', 'on back of', 'over',
    'painted on', 'parked on', 'part of', 'playing', 'riding', 'says', 'sitting on', 'standing on',
    'to', 'under', 'using', 'walking in', 'walking on', 'watching', 'wearing', 'wears', 'with'
]

def rel_name(idx):
    idx = int(idx)
    if 0 <= idx < len(VG_REL_CLASSES):
        return VG_REL_CLASSES[idx]
    return f'rel_{idx}'

def to_np(x):
    if x is None:
        return None
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def has_field(box, name):
    return hasattr(box, 'has_field') and box.has_field(name)

def get_field_np(box, name, default=None):
    if not has_field(box, name):
        return default
    return to_np(box.get_field(name))

def box_fields(box):
    return sorted(list(getattr(box, 'extra_fields', {}).keys()))

def auto_find(name):
    candidates = list(REPO_ROOT.rglob(name))
    if not candidates:
        return None
    candidates = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]

def scalar_summary(v):
    """Robustly summarize result_dict entries, including nested lists from per-class collections."""
    if torch.is_tensor(v):
        v = v.detach().cpu().numpy()
    if isinstance(v, np.ndarray):
        if v.size == 0:
            return np.nan
        return float(np.nanmean(v.astype(float)))
    if isinstance(v, (int, float, np.number)):
        return float(v)
    if isinstance(v, list):
        flat = []
        stack = list(v)
        while stack:
            item = stack.pop()
            if isinstance(item, (list, tuple)):
                stack.extend(item)
            elif torch.is_tensor(item):
                arr = item.detach().cpu().numpy().reshape(-1)
                flat.extend([float(x) for x in arr])
            elif isinstance(item, np.ndarray):
                flat.extend([float(x) for x in item.reshape(-1)])
            elif isinstance(item, (int, float, np.number)):
                flat.append(float(item))
        return float(np.nanmean(flat)) if flat else np.nan
    return np.nan

def show_table(df, title=None, max_rows=100):
    if title:
        print('\n' + title)
    display(df.head(max_rows) if len(df) > max_rows else df)


In [ ]:
eval_path = Path(EVAL_RESULTS_PATH) if EVAL_RESULTS_PATH else auto_find('eval_results.pytorch')
result_path = Path(RESULT_DICT_PATH) if RESULT_DICT_PATH else auto_find('result_dict.pytorch')
print('eval_results:', eval_path)
print('result_dict:', result_path)
assert eval_path and eval_path.exists(), '找不到 eval_results.pytorch，请手动填写 EVAL_RESULTS_PATH'
assert result_path and result_path.exists(), '找不到 result_dict.pytorch，请手动填写 RESULT_DICT_PATH'

eval_blob = torch.load(str(eval_path), map_location='cpu')
result_dict = torch.load(str(result_path), map_location='cpu')
groundtruths = eval_blob['groundtruths']
predictions = eval_blob['predictions']
print('num images:', len(predictions), 'num gt:', len(groundtruths))
print('prediction fields:', box_fields(predictions[0]))
print('groundtruth fields:', box_fields(groundtruths[0]))
print('result_dict keys:', sorted(result_dict.keys()))

## 1. 汇总指标与谓词级 mR

In [ ]:
summary_rows = []
for key, value in result_dict.items():
    if isinstance(value, dict) and set(value.keys()) >= {20, 50, 100}:
        row = {'metric': key}
        for k in [20, 50, 100]:
            row[f'@{k}'] = scalar_summary(value[k])
        summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows).sort_values('metric')
show_table(summary_df, 'Metric summary')


In [ ]:
def predicate_recall_table(prefix='predcls_mean_recall_list'):
    rows = []
    if prefix not in result_dict:
        print('missing', prefix)
        return pd.DataFrame()
    for k in [20, 50, 100]:
        recalls = result_dict[prefix].get(k, [])
        for i, r in enumerate(recalls, start=1):
            rows.append({'rel_id': i, 'predicate': rel_name(i), 'K': k, 'recall': scalar_summary(r)})
    return pd.DataFrame(rows)

mr_key = next((k for k in result_dict if k.endswith('_mean_recall_list')), 'predcls_mean_recall_list')
print('Using mean recall key:', mr_key)
mr_df = predicate_recall_table(mr_key)
pivot_mr = mr_df.pivot(index=['rel_id', 'predicate'], columns='K', values='recall').reset_index()
pivot_mr = pivot_mr.sort_values(100, ascending=True)
show_table(pivot_mr, 'Predicate mean recall, sorted by mR@100')

if len(pivot_mr):
    ax = pivot_mr.tail(20).sort_values(100).plot.barh(x='predicate', y=100, legend=False, title='Top 20 predicates by mR@100')
    ax.set_xlabel('recall')
    plt.show()
    ax = pivot_mr.head(20).sort_values(100, ascending=False).plot.barh(x='predicate', y=100, legend=False, title='Worst 20 predicates by mR@100')
    ax.set_xlabel('recall')
    plt.show()


## 2. 预测结果展开为 pair-level 表

In [ ]:
def build_prediction_table(topn_per_image=None):
    rows = []
    for image_idx, pred in enumerate(predictions):
        rel_pairs = get_field_np(pred, 'rel_pair_idxs')
        rel_scores = get_field_np(pred, 'pred_rel_scores')
        if rel_pairs is None or rel_scores is None or len(rel_pairs) == 0:
            continue
        pred_labels = get_field_np(pred, 'pred_labels')
        pred_obj_scores = get_field_np(pred, 'pred_scores')
        rel_logits = get_field_np(pred, 'pred_rel_logit')
        relationness = get_field_np(pred, 'relationness_scores')
        fg_scores = rel_scores[:, 1:]
        top_rel = fg_scores.argmax(axis=1) + 1
        top_score = fg_scores.max(axis=1)
        order = np.arange(len(rel_pairs))
        if topn_per_image is not None:
            order = order[:topn_per_image]
        for rank, j in enumerate(order):
            s, o = rel_pairs[j]
            row = {
                'image_idx': image_idx,
                'rank': rank + 1,
                'sub_idx': int(s),
                'obj_idx': int(o),
                'sub_cls': int(pred_labels[s]) if pred_labels is not None else -1,
                'obj_cls': int(pred_labels[o]) if pred_labels is not None else -1,
                'sub_score': float(pred_obj_scores[s]) if pred_obj_scores is not None else np.nan,
                'obj_score': float(pred_obj_scores[o]) if pred_obj_scores is not None else np.nan,
                'pred_rel': int(top_rel[j]),
                'pred_rel_name': rel_name(top_rel[j]),
                'pred_rel_score': float(top_score[j]),
                'relationness': float(relationness[j]) if relationness is not None else np.nan,
            }
            if rel_logits is not None:
                row['pred_rel_logit_max'] = float(rel_logits[j, 1:].max())
                row['pred_rel_logit_bg'] = float(rel_logits[j, 0])
            rows.append(row)
    return pd.DataFrame(rows)

pred_df = build_prediction_table()
show_table(pred_df.head(20), 'Prediction pairs: first 20 rows')
display(pred_df.describe(include='all'))


In [ ]:
if len(pred_df):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    pred_df['pred_rel_score'].hist(ax=axes[0], bins=50)
    axes[0].set_title('predicate max probability')
    if pred_df['relationness'].notna().any():
        pred_df['relationness'].hist(ax=axes[1], bins=50)
        axes[1].set_title('relationness')
        pred_df.plot.scatter(x='relationness', y='pred_rel_score', alpha=0.1, ax=axes[2])
        axes[2].set_title('relationness vs predicate score')
    plt.tight_layout()
    plt.show()

    pred_count = pred_df.groupby(['pred_rel', 'pred_rel_name']).size().reset_index(name='num_predictions')
    pred_count = pred_count.sort_values('num_predictions', ascending=False)
    show_table(pred_count.head(30), 'Most frequent predicted predicates')


## 3. Top-K 中非 GT pair 数量

这里的非 GT pair 指：预测 Top-K 中的 `(subject_idx, object_idx)` 不在当前图的 GT `relation_tuple[:, :2]` 里。它能直接反映负样本 pair 是否挤占了真实关系 pair 的排序空间。


In [ ]:
def build_topk_pair_table(ks=(20, 50, 100)):
    rows = []
    for image_idx, (gt, pred) in enumerate(zip(groundtruths, predictions)):
        gt_rels = get_field_np(gt, 'relation_tuple')
        rel_pairs = get_field_np(pred, 'rel_pair_idxs')
        if gt_rels is None or rel_pairs is None:
            continue
        gt_pairs = {tuple(map(int, x[:2])) for x in gt_rels.tolist()}
        for k in ks:
            top_pairs = [tuple(map(int, x)) for x in rel_pairs[:k].tolist()]
            non_gt = [p for p in top_pairs if p not in gt_pairs]
            gt_hit_pairs = [p for p in top_pairs if p in gt_pairs]
            rows.append({
                'image_idx': image_idx,
                'K': k,
                'num_gt_relations': len(gt_rels),
                'num_gt_pairs': len(gt_pairs),
                'num_pred_pairs_in_topk': len(top_pairs),
                'num_non_gt_pairs': len(non_gt),
                'non_gt_pair_ratio': len(non_gt) / max(len(top_pairs), 1),
                'num_gt_pairs_covered': len(set(gt_hit_pairs)),
                'gt_pair_coverage': len(set(gt_hit_pairs)) / max(len(gt_pairs), 1),
            })
    return pd.DataFrame(rows)

topk_pair_df = build_topk_pair_table()
show_table(topk_pair_df.head(30), 'Per-image Top-K non-GT pair count')

if len(topk_pair_df):
    topk_summary = topk_pair_df.groupby('K').agg(
        images=('image_idx', 'count'),
        mean_non_gt_pairs=('num_non_gt_pairs', 'mean'),
        median_non_gt_pairs=('num_non_gt_pairs', 'median'),
        mean_non_gt_ratio=('non_gt_pair_ratio', 'mean'),
        mean_gt_pair_coverage=('gt_pair_coverage', 'mean'),
    ).reset_index()
    show_table(topk_summary, 'Top-K non-GT pair summary')

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    topk_pair_df.boxplot(column='num_non_gt_pairs', by='K', ax=axes[0])
    axes[0].set_title('Non-GT pairs in Top-K')
    axes[0].figure.suptitle('')
    topk_pair_df.boxplot(column='gt_pair_coverage', by='K', ax=axes[1])
    axes[1].set_title('GT pair coverage in Top-K')
    axes[1].figure.suptitle('')
    plt.tight_layout()
    plt.show()


## 4. GT 谓词在对应 pair 的谓词排序中排第几

对每个 GT triplet `(sub, obj, rel)`，找到预测结果里的同一个 pair，然后看 GT 谓词在该 pair 的 `pred_rel_scores[:, 1:]` 里排第几。

如果 GT 谓词不是第 1，表格会列出排在它前面的谓词，按分数从高到低排列。


In [ ]:
def build_gt_pair_table():
    rows = []
    for image_idx, (gt, pred) in enumerate(zip(groundtruths, predictions)):
        gt_rels = get_field_np(gt, 'relation_tuple')
        rel_pairs = get_field_np(pred, 'rel_pair_idxs')
        rel_scores = get_field_np(pred, 'pred_rel_scores')
        relationness = get_field_np(pred, 'relationness_scores')
        if gt_rels is None or rel_pairs is None or rel_scores is None:
            continue
        pair_to_pred = {tuple(map(int, p)): idx for idx, p in enumerate(rel_pairs.tolist())}
        for gt_idx, (s, o, r) in enumerate(gt_rels.astype(int).tolist()):
            j = pair_to_pred.get((s, o))
            if j is None:
                rows.append({
                    'image_idx': image_idx, 'gt_idx': gt_idx, 'sub_idx': s, 'obj_idx': o,
                    'gt_rel': r, 'gt_rel_name': rel_name(r), 'found_pair': False,
                    'pair_rank': np.nan, 'pred_rel': -1, 'pred_rel_name': 'missing_pair',
                    'pred_rel_score': np.nan, 'gt_rel_score': np.nan,
                    'gt_predicate_rank_in_pair': np.nan, 'blocked_by_predicates': '',
                    'relationness': np.nan, 'correct_top1': False,
                })
                continue
            scores = rel_scores[j, 1:]
            sorted_fg = np.argsort(-scores) + 1
            pred_r = int(sorted_fg[0])
            if r > 0 and r in sorted_fg:
                gt_rank = int(np.where(sorted_fg == r)[0][0] + 1)
                blockers = [
                    f'{rel_name(rr)}:{float(rel_scores[j, rr]):.4f}'
                    for rr in sorted_fg[:gt_rank - 1]
                ]
                gt_score = float(rel_scores[j, r])
            else:
                gt_rank = np.nan
                blockers = []
                gt_score = np.nan
            rows.append({
                'image_idx': image_idx,
                'gt_idx': gt_idx,
                'sub_idx': s,
                'obj_idx': o,
                'gt_rel': r,
                'gt_rel_name': rel_name(r),
                'found_pair': True,
                'pair_rank': int(j + 1),
                'pred_rel': pred_r,
                'pred_rel_name': rel_name(pred_r),
                'pred_rel_score': float(scores.max()),
                'gt_rel_score': gt_score,
                'gt_predicate_rank_in_pair': gt_rank,
                'blocked_by_predicates': ' | '.join(blockers),
                'relationness': float(relationness[j]) if relationness is not None else np.nan,
                'correct_top1': pred_r == r,
            })
    return pd.DataFrame(rows)

gt_pair_df = build_gt_pair_table()
show_table(gt_pair_df.head(30), 'GT predicate rank examples')
print('GT pair top-1 predicate accuracy:', gt_pair_df['correct_top1'].mean() if len(gt_pair_df) else np.nan)

if len(gt_pair_df):
    per_predicate_rank = gt_pair_df.groupby(['gt_rel', 'gt_rel_name']).agg(
        count=('gt_rel', 'size'),
        found_pair_rate=('found_pair', 'mean'),
        top1_acc=('correct_top1', 'mean'),
        mean_pair_rank=('pair_rank', 'mean'),
        median_pair_rank=('pair_rank', 'median'),
        mean_gt_predicate_rank=('gt_predicate_rank_in_pair', 'mean'),
        median_gt_predicate_rank=('gt_predicate_rank_in_pair', 'median'),
        mean_gt_score=('gt_rel_score', 'mean'),
        mean_relationness=('relationness', 'mean'),
    ).reset_index().sort_values(['top1_acc', 'mean_gt_predicate_rank', 'count'], ascending=[True, True, False])
    show_table(per_predicate_rank, 'Per-predicate GT rank and top-1 accuracy')

    hard_cases = gt_pair_df[(gt_pair_df['found_pair']) & (~gt_pair_df['correct_top1'])].copy()
    hard_cases = hard_cases.sort_values(['gt_predicate_rank_in_pair', 'gt_rel_score'], ascending=[False, True])
    show_table(hard_cases[['image_idx', 'pair_rank', 'gt_rel_name', 'gt_predicate_rank_in_pair', 'gt_rel_score', 'pred_rel_name', 'pred_rel_score', 'blocked_by_predicates']].head(100),
               'GT predicate is not rank-1: blockers before GT predicate')


## 5. 每个谓词在 Top-20 / Top-50 / Top-100 的召回

这里直接根据预测排序后的 graph-constrained Top-K 计算：GT pair 出现在 Top-K 且该 pair 的 top-1 谓词等于 GT 谓词，记为召回。


In [ ]:
def build_predicate_recall_from_predictions(ks=(20, 50, 100)):
    rows = []
    for rel_id in range(1, len(VG_REL_CLASSES)):
        rel_rows = gt_pair_df[gt_pair_df['gt_rel'] == rel_id]
        if len(rel_rows) == 0:
            continue
        row = {'gt_rel': rel_id, 'gt_rel_name': rel_name(rel_id), 'count': len(rel_rows)}
        for k in ks:
            hit = ((rel_rows['pair_rank'] <= k) & (rel_rows['correct_top1'])).sum()
            row[f'R@{k}'] = float(hit) / float(len(rel_rows))
            row[f'hit@{k}'] = int(hit)
        rows.append(row)
    return pd.DataFrame(rows)

pred_recall_df = build_predicate_recall_from_predictions()
if len(pred_recall_df):
    pred_recall_df = pred_recall_df.sort_values('R@100')
    show_table(pred_recall_df, 'Per-predicate recall from prediction ranking')

    ax = pred_recall_df.head(20).sort_values('R@100', ascending=False).plot.barh(
        x='gt_rel_name', y='R@100', legend=False, title='Worst 20 predicate recall@100 from predictions'
    )
    ax.set_xlabel('recall@100')
    plt.show()


In [ ]:
if len(gt_pair_df):
    conf = gt_pair_df[gt_pair_df['found_pair']].groupby(
        ['gt_rel_name', 'pred_rel_name']
    ).size().reset_index(name='count').sort_values('count', ascending=False)
    wrong_conf = conf[conf['gt_rel_name'] != conf['pred_rel_name']]
    display(wrong_conf.head(50))

    if gt_pair_df['relationness'].notna().any():
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        gt_pair_df.boxplot(column='relationness', by='correct_top1', ax=axes[0])
        axes[0].set_title('relationness by correctness')
        axes[0].figure.suptitle('')
        gt_pair_df.boxplot(column='gt_rel_score', by='correct_top1', ax=axes[1])
        axes[1].set_title('GT predicate score by correctness')
        axes[1].figure.suptitle('')
        plt.tight_layout()
        plt.show()

## 6. 图像级失败案例排序

In [ ]:
if len(gt_pair_df):
    image_df = gt_pair_df.groupby('image_idx').agg(
        num_gt=('gt_rel', 'size'),
        num_found_pair=('found_pair', 'sum'),
        top1_hits=('correct_top1', 'sum'),
        top1_acc=('correct_top1', 'mean'),
        mean_gt_score=('gt_rel_score', 'mean'),
        mean_relationness=('relationness', 'mean'),
    ).reset_index()
    image_df['miss_top1'] = image_df['num_gt'] - image_df['top1_hits']
    image_df = image_df.sort_values(['top1_acc', 'num_gt'], ascending=[True, False])
    display(image_df.head(50))


## 7. 单张图像展开查看

In [ ]:
IMAGE_IDX = int(image_df.iloc[0]['image_idx']) if 'image_df' in globals() and len(image_df) else 0
print('IMAGE_IDX =', IMAGE_IDX)

display(gt_pair_df[gt_pair_df['image_idx'] == IMAGE_IDX].sort_values(['correct_top1', 'rank_of_gt_rel']))
display(pred_df[pred_df['image_idx'] == IMAGE_IDX].head(100))